<a href="https://colab.research.google.com/github/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution/blob/main/notebooks/07_cross_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5-fold cross-validation and paired t-tests

Manuscript section 3.8.1: *"5-fold cross-validation is applied to ensure the model's stability across different data subsets…
a paired t-test (multi-signal vs single-signal F1-score) is used to determine if the performance differences before and after compression
are statistically significant (p < 0.05)."*

All 19,820 images of the cleaned splits are pooled and divided into 5 folds, balanced by class. Near-duplicates are grouped across the
**whole** pool and each group stays in one fold, so no copy of a test image is ever trained on. For each fold, the other four folds are split
into training and validation (15% of each class), ResNet-50 and EfficientNet-B0 are trained and calibrated, the watermark threshold and confidence
are set on that fold's validation images, and the fold's ~3,960 test images go through all 9 conditions and all four systems.

Across the 5 folds this gives every result a mean and a spread, and five paired F1 values per condition for the t-tests.

**Runtime > Change runtime type > GPU** (T4 or L4). Roughly **1 hour per fold, about 5 hours in total.** Everything is saved to Google Drive
(`thesis-data/cv/`) as it finishes, so if Colab disconnects just run sections 1-2 again and continue with the next fold: finished steps are skipped.

## 1. Get the code

In [1]:
%cd /content
!rm -rf repo
!git clone https://github.com/Cheponke/Transformation-Aware-Reliability-Modeling-for-Multi-Signal-Synthetic-Image-Attribution.git repo
%cd /content/repo
!pip install -q -e ".[train,provenance,dev]"
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (training will be very slow)")

/content
Cloning into 'repo'...
remote: Enumerating objects: 479, done.
remote: Counting objects: 100% (243/243), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 479 (delta 143), reused 95 (delta 53), pack-reused 236 (from 1)
Receiving objects: 100% (479/479), 6.90 MiB | 12.56 MiB/s, done.
Resolving deltas: 100% (243/243), done.
/content/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for attribution (pyproject.toml) ... done
GPU: Tesla T4


## 2. Restore the cleaned data and link the results folder on Google Drive

In [2]:
from google.colab import drive
import os
drive.mount("/content/drive")
D = "/content/drive/MyDrive/thesis-data"
ZIP = f"{D}/harmonized.zip"
if not os.path.exists(ZIP):
    raise FileNotFoundError(f"{ZIP} not found: run notebook 01c first")
!unzip -qo $ZIP -d .
!wc -l data/harmonized/train.csv data/harmonized/val.csv data/harmonized/test.csv
!mkdir -p $D/cv && rm -rf outputs/cv && ln -s $D/cv outputs/cv

Mounted at /content/drive
  13849 data/harmonized/train.csv
   2981 data/harmonized/val.csv
   2993 data/harmonized/test.csv
  19823 total


## 3. Run the folds
One cell per fold, about an hour each. The first cell also builds the fold assignment (`folds.csv`, a few minutes) and prints the fold sizes.
If the session drops, rerun sections 1-2 and then the fold you were on: it picks up where it stopped.

In [3]:
!python scripts/cross_validate.py --folds 0

folds: grouping near-duplicates among 19820 images (dHash distance <= 5)
folds: 19702 groups, 106 with more than one image (224 images), largest 3
  fold 0: 3964 images (1981 human / 1983 AI)
  fold 1: 3964 images (1981 human / 1983 AI)
  fold 2: 3964 images (1981 human / 1983 AI)
  fold 3: 3964 images (1982 human / 1982 AI)
  fold 4: 3964 images (1982 human / 1982 AI)
== fold 0: 13477 train / 2379 val / 3964 test
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 172MB/s]
  resnet50: training on cuda: 13477 train / 2379 val images
epoch 1: train_loss=0.3304 val_loss=0.2060 val_acc=0.9172
epoch 2: train_loss=0.2069 val_loss=0.1701 val_acc=0.9344
epoch 3: train_loss=0.1454 val_loss=0.1808 val_acc=0.9269
epoch 4: train_loss=0.1231 val_loss=0.1758 val_acc=0.9357
epoch 5: train_loss=0.1051 val_loss=0.2275 val_acc=0.9185
epoch 6: train_loss=0.0758 val_loss=0.1911 val_acc=0.9348
e

In [ ]:
!python scripts/cross_validate.py --folds 1

In [ ]:
!python scripts/cross_validate.py --folds 2

In [ ]:
!python scripts/cross_validate.py --folds 3

In [ ]:
!python scripts/cross_validate.py --folds 4

## 4. Results across the 5 folds
**mean ± sd** over the folds for each system and condition. The t-tests compare multi-signal F1 with each single-signal F1, pairing the
five folds (each fold's systems saw exactly the same test images). **p** is the paired t-test the manuscript specifies;
**p (Holm)** additionally corrects for running many tests at once (2 backbones × 9 conditions × 3 comparisons), which a panel may ask about.
A system that never flags an AI image (provenance after re-encoding) has F1 = 0. Where the difference is identical in every fold
(e.g. both systems perfect on clean images) the t-test is undefined and marked as such.

In [ ]:
!python scripts/cross_validate.py --aggregate-only
import json, pandas as pd
pd.set_option("display.width", 220)
ms = pd.read_csv("outputs/cv/summary_mean_sd.csv")
order = ms.condition.unique()
fmt = lambda m, s: "-" if pd.isna(m) else (f"{m:.3f}" if pd.isna(s) else f"{m:.3f} ± {s:.3f}")
for arch in ms.arch.unique():
    for metric in ["f1", "accuracy", "recall", "fpr", "coverage", "ece"]:
        t = ms[ms.arch == arch].assign(v=lambda d: [fmt(m, s) for m, s in zip(d[f"{metric}_mean"], d[f"{metric}_sd"])])
        print(f"\n{arch} | {metric} (mean ± sd over folds)")
        display(t.pivot(index="condition", columns="system", values="v").loc[order, ["provenance_only", "watermark_only", "cnn_only", "multi_signal"]])

In [ ]:
sig = pd.read_csv("outputs/cv/significance.csv")
print(json.load(open("outputs/cv/cv_report.json")))
cols = ["condition", "comparison", "mean_f1_multi_signal", "mean_f1_single", "mean_difference", "sd_difference",
        "t_statistic", "p_value", "p_holm", "significant_p05", "significant_holm", "note"]
for arch in sig.arch.unique():
    print(f"\n{arch}: paired t-tests, multi-signal vs single-signal F1 (5 folds)")
    display(sig[sig.arch == arch][cols].round(4))

## 5. Save it
File > Save a copy in GitHub. The fold assignment, per-fold summaries, mean ± sd table and t-tests are on Drive in `thesis-data/cv/`
and are collected by notebook 06.